In [ ]:
import os
import pandas as pd
from transformers import pipeline
from dotenv import load_dotenv

load_dotenv()


# Hugging Face
from huggingface_hub import login

login(token="#")


In [4]:
"""Function cleans string and leaves only the song lyrics we want"""
def clean_lyrics(raw_lyrics):
    #Get rid of information shoved at beginning of string leaving only the lyrics of the song
    starter = ']' #This always finds the first closed bracket which is always the closing bracket for '[Intro]'
    position = raw_lyrics.find(starter)
    lyrics = raw_lyrics[position+1:]

    # Remove section headers like [Chorus], [Verse 1], etc.
    lyrics = re.sub(r"\[.*?\]", "", lyrics)

    # Remove extra whitespace, newlines
    lyrics = re.sub(r"\n+", " ", lyrics)
    lyrics = re.sub(r"\s+", " ", lyrics)

    # Optionally remove punctuation
    lyrics = re.sub(r"[^\w\s']", "", lyrics)

    # Lowercase
    lyrics = lyrics.lower()

    return lyrics

"""Function searchs song and returns string containing only the lyrics"""
def song_search(song,artist):
    genius = lyricsgenius.Genius("rPVs-pFT7GhBfTxhpu-ISnNCcvCsbRt8wIwhkkEXovrADgSBQfkndQW7Ge22R5Ts", timeout=60)
    song = genius.search_song(song, artist)
    lyrics = song.lyrics
    
    cleaned_lyrics = clean_lyrics(lyrics)
    return cleaned_lyrics

In [5]:
# load in lyrics
df = pd.read_csv('lyrics_list.txt', delimiter=',',header=None)
df.columns = ['lyrics','Nan']
df = df.drop(labels='Nan',axis=1)

In [6]:

# We use facebook/bart-large-mnli for zero-shot theme detection
# then format the results into a prose summary — no GPT needed
zero_shot = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=0
)

THEMES = [
    "love and romance", "heartbreak and loss", "identity and self-discovery",
    "rebellion and resistance", "nostalgia and memory", "depression and mental health",
    "social commentary", "celebration and joy", "spirituality and faith",
    "ambition and success", "loneliness and isolation", "death and mortality",
]

THRESHOLD = 0.20

def generate_prose_summary(lyrics: str) -> str:
    words = lyrics.split()
    chunk_size = 400
    chunks = [" ".join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]

    # Accumulate scores across chunks
    score_map = {theme: 0.0 for theme in THEMES}
    for chunk in chunks:
        result = zero_shot(chunk, THEMES, multi_label=True)
        for label, score in zip(result["labels"], result["scores"]):
            score_map[label] += score

    # Average scores across chunks
    n = len(chunks)
    avg_scores = {t: score_map[t] / n for t in THEMES}

    # Get themes above threshold, sorted by confidence
    detected = sorted(
        [(t, s) for t, s in avg_scores.items() if s >= THRESHOLD],
        key=lambda x: -x[1]
    )

    # Always keep at least the top theme
    if not detected:
        detected = [(max(avg_scores, key=avg_scores.get), 1.0)]

    # Format into a natural prose summary
    theme_names = [t for t, s in detected]
    if len(theme_names) == 1:
        theme_str = theme_names[0]
    elif len(theme_names) == 2:
        theme_str = f"{theme_names[0]} and {theme_names[1]}"
    else:
        theme_str = ", ".join(theme_names[:-1]) + f", and {theme_names[-1]}"

    summary = f"This song explores themes of {theme_str}."
    return summary


summaries = []
for i, row in df.iterrows():
    summary = generate_prose_summary(row["lyrics"])
    summaries.append(summary)
    if i % 50 == 0:
        print(f"Processed {i}/{len(df)} — {summary}")

df["summary"] = summaries
df.to_json("lyrics_with_summaries.json", orient="records", lines=True)
print("Done. Dataset saved to lyrics_with_summaries.json")

Device set to use cuda:0


Processed 0/909 — This song explores themes of social commentary, heartbreak and loss, love and romance, identity and self-discovery, nostalgia and memory, depression and mental health, ambition and success, rebellion and resistance, spirituality and faith, loneliness and isolation, celebration and joy, and death and mortality.


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Processed 50/909 — This song explores themes of rebellion and resistance, social commentary, ambition and success, death and mortality, identity and self-discovery, heartbreak and loss, nostalgia and memory, depression and mental health, loneliness and isolation, and spirituality and faith.
Processed 100/909 — This song explores themes of nostalgia and memory, heartbreak and loss, identity and self-discovery, social commentary, spirituality and faith, depression and mental health, love and romance, loneliness and isolation, death and mortality, celebration and joy, rebellion and resistance, and ambition and success.
Processed 150/909 — This song explores themes of celebration and joy, death and mortality, rebellion and resistance, ambition and success, social commentary, nostalgia and memory, identity and self-discovery, love and romance, depression and mental health, loneliness and isolation, spirituality and faith, and heartbreak and loss.
Processed 200/909 — This song explores theme

In [8]:
import os
import torch
import pandas as pd
from torch.utils.data import Dataset
from transformers import (
    BartTokenizer,
    BartForConditionalGeneration,
    Seq2SeqTrainingArguments,   # replaces TrainingArguments
    Seq2SeqTrainer,             # replaces Trainer
    DataCollatorForSeq2Seq,
)
from sklearn.model_selection import train_test_split
from dotenv import load_dotenv

load_dotenv()

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:512"

# ── Config ────────────────────────────────────────────────────────────────────
MODEL_NAME      = "facebook/bart-large-cnn"  # pretrained on summarization already
MAX_INPUT       = 1024   # BART's maximum input length
MAX_TARGET      = 128    # summaries are short
BATCH_SIZE      = 2
GRAD_ACCUM      = 8      # effective batch size = 16
EPOCHS          = 6
LR              = 2e-5

# ── Dataset class ─────────────────────────────────────────────────────────────
class LyricsSummaryDataset(Dataset):
    def __init__(self, lyrics_list, summaries_list, tokenizer):
        self.inputs  = []
        self.targets = []

        for lyrics, summary in zip(lyrics_list, summaries_list):
            # Prefix helps BART understand the task
            prompt = f"summarize the themes of these lyrics: {lyrics}"

            model_inputs = tokenizer(
                prompt,
                max_length=MAX_INPUT,
                padding="max_length",
                truncation=True,
                return_tensors="pt",
            )
            labels = tokenizer(
                summary,
                max_length=MAX_TARGET,
                padding="max_length",
                truncation=True,
                return_tensors="pt",
            )

            # Replace padding token id in labels with -100 so loss ignores them
            label_ids = labels["input_ids"].squeeze()
            label_ids[label_ids == tokenizer.pad_token_id] = -100

            self.inputs.append({
                "input_ids":      model_inputs["input_ids"].squeeze(),
                "attention_mask": model_inputs["attention_mask"].squeeze(),
            })
            self.targets.append(label_ids)

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        return {**self.inputs[idx], "labels": self.targets[idx]}


# ── Load and split data ───────────────────────────────────────────────────────
df = pd.read_json("lyrics_with_summaries.json", lines=True)
df = df.dropna(subset=["lyrics", "summary"]).reset_index(drop=True)

train_df, val_df = train_test_split(df, test_size=0.15, random_state=42)

tokenizer = BartTokenizer.from_pretrained(MODEL_NAME)

train_dataset = LyricsSummaryDataset(
    train_df["lyrics"].tolist(),
    train_df["summary"].tolist(),
    tokenizer
)
val_dataset = LyricsSummaryDataset(
    val_df["lyrics"].tolist(),
    val_df["summary"].tolist(),
    tokenizer
)

# ── Model ─────────────────────────────────────────────────────────────────────
model = BartForConditionalGeneration.from_pretrained(MODEL_NAME)
model.gradient_checkpointing_enable()

# ── Metrics ───────────────────────────────────────────────────────────────────
import numpy as np
from nltk.translate.bleu_score import corpus_bleu

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)

    # Replace -100 in labels (padding) before decoding
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # BLEU score — measures how close generated summaries are to targets
    references = [[ref.split()] for ref in decoded_labels]
    hypotheses = [pred.split() for pred in decoded_preds]
    bleu = corpus_bleu(references, hypotheses)

    return {"bleu": round(bleu, 4)}

# ── Training ──────────────────────────────────────────────────────────────────
training_args = Seq2SeqTrainingArguments(
    output_dir="./lyric-theme-bart",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="bleu",
    bf16=True,
    gradient_checkpointing=True,
    dataloader_num_workers=0,
    predict_with_generate=True,   # needed for generative models
    logging_steps=25,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

import gc
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM free:  {torch.cuda.mem_get_info()[0] / 1e9:.2f} GB")
print(f"VRAM total: {torch.cuda.mem_get_info()[1] / 1e9:.2f} GB")

trainer.train()
trainer.save_model("./lyric-theme-bart/best")
tokenizer.save_pretrained("./lyric-theme-bart/best")
print("Training complete.")

C:\Users\Monado\AppData\Local\Temp\ipykernel_23980\2961542096.py:136: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


VRAM free:  4.10 GB
VRAM total: 8.59 GB


Epoch,Training Loss,Validation Loss,Bleu
1,1.544200,0.445070,0.402700
2,0.355600,0.389257,0.417400
3,0.297800,0.407686,0.446000
4,0.262800,0.352012,0.444700
5,0.231400,0.383917,0.441700
6,0.207300,0.400533,0.451000


c:\Users\Monado\AppData\Local\Programs\Python\Python311\Lib\site-packages\transformers\modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 142, 'min_length': 56, 'early_stopping': True, 'num_beams': 4, 'length_penalty': 2.0, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(
There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


Training complete.


In [9]:
# This code allowed me to pull the results from my trained model

# Load from the best checkpoint that was saved during training
LOCAL_PATH = "./lyric-theme-bart/best"

model = BartForConditionalGeneration.from_pretrained(LOCAL_PATH)
tokenizer = BartTokenizer.from_pretrained(LOCAL_PATH)

print("Model loaded successfully")

c:\Users\Monado\AppData\Local\Programs\Python\Python311\Lib\site-packages\transformers\models\bart\configuration_bart.py:177: UserWarning: Please make sure the config includes `forced_bos_token_id=0` in future versions. The config can simply be saved and uploaded again to be fixed.
  warnings.warn(


Model loaded successfully


In [10]:
# This code is how I saved my model weights to hugging face
HF_REPO = "gpecorino/lyric-theme-bart"

model.push_to_hub(HF_REPO)
tokenizer.push_to_hub(HF_REPO)
print(f"Model saved to https://huggingface.co/{HF_REPO}")

c:\Users\Monado\AppData\Local\Programs\Python\Python311\Lib\site-packages\transformers\modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'forced_bos_token_id': 0}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(
Processing Files (1 / 1): 100%|██████████| 1.63GB / 1.63GB, 18.6MB/s  
New Data Upload: 100%|██████████| 1.63GB / 1.63GB, 18.6MB/s  
c:\Users\Monado\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Monado\.cache\huggingface\hub\models--gpecorino--lyric-theme-bart. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE

Model saved to https://huggingface.co/gpecorino/lyric-theme-bart


In [11]:
# This code is how I pull my saved weights from hugging face
HF_REPO = "gpecorino/lyric-theme-bart"

model = BartForConditionalGeneration.from_pretrained(HF_REPO)
tokenizer = BartTokenizer.from_pretrained(HF_REPO)
model.eval()

BartForConditionalGeneration(
  (model): BartModel(
    (shared): BartScaledWordEmbedding(50264, 1024, padding_idx=1)
    (encoder): BartEncoder(
      (embed_tokens): BartScaledWordEmbedding(50264, 1024, padding_idx=1)
      (embed_positions): BartLearnedPositionalEmbedding(1026, 1024)
      (layers): ModuleList(
        (0-11): 12 x BartEncoderLayer(
          (self_attn): BartAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias=True)
        

In [12]:
from transformers import BartTokenizer, BartForConditionalGeneration
import torch

# model = BartForConditionalGeneration.from_pretrained("./lyric-theme-bart/best")
# tokenizer = BartTokenizer.from_pretrained("./lyric-theme-bart/best")
# model.eval()

def summarize_themes(lyrics: str) -> str:
    prompt = f"summarize the themes of these lyrics: {lyrics}"
    inputs = tokenizer(
        prompt,
        max_length=1024,
        truncation=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        summary_ids = model.generate(
            inputs["input_ids"],
            max_new_tokens=80,
            min_length=15,
            num_beams=4,          # beam search for better quality
            length_penalty=1.5,   # encourages slightly longer summaries
            early_stopping=True,
            no_repeat_ngram_size=3  # prevents repetitive phrases
        )

    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)


# Example
lyrics = """
I keep your photograph, I know it serves me well
When I close my eyes, I can almost smell your perfume
"""
print(summarize_themes(lyrics))
# → "This song explores themes of heartbreak and longing,
#    reflecting on a lost relationship through nostalgic memory."

This song explores themes of nostalgia and memory, love and romance, social commentary, identity and self-discovery, heartbreak and loss, depression and mental health, rebellion and resistance, and death and mortality.
